# 02 -- model training (v3)

trains the n-tuple network board evaluator via temporal difference learning (td) reinforcement learning across 100,000 self-play games.

features in version 3:
- n-tuple network architecture: 4 base 5-tuples x 8 d4 dihedral symmetries = 32 symmetric tuples
- total parameters: 4 x 16^5 = 4,194,304 float32 weights (16.77 mb binary table)
- afterstate temporal difference learning: delta = r + V(S'_next) - V(S')
- exact symmetry-normalized gradient updates
- direct binary export: `tuples_v3.bin` matching js/nn.js memory layout bit-for-bit
- zero heavy dependencies: runs on pure numpy + optional numba jit for high speed
- packaged into `02_model_training_output.zip`

In [ ]:
import os
import shutil
import json
import time
import random
import numpy as np

# check for numba acceleration
try:
    import numba
    from numba import njit, prange
    has_numba = True
    print(f'numba acceleration available! version: {numba.__version__}')
except ImportError:
    has_numba = False
    print('numba not installed, running with vectorized numpy / pure python')
    def njit(*args, **kwargs):
        def decorator(f):
            return f
        return decorator

## game engine and afterstate transitions

In [ ]:
def slide_line(line):
    compact = [x for x in line if x != 0]
    score = 0
    res = []
    i = 0
    while i < len(compact):
        if i + 1 < len(compact) and compact[i] == compact[i + 1]:
            new_exp = compact[i] + 1
            res.append(new_exp)
            score += (1 << new_exp)
            i += 2
        else:
            res.append(compact[i])
            i += 1
    while len(res) < 4:
        res.append(0)
    return res, score

def move_board(b, direction):
    # b is 16-element list/array of tile exponents (0..15)
    new_b = list(b)
    total_score = 0
    moved = False

    for i in range(4):
        if direction == 0:   line = [b[0*4 + i], b[1*4 + i], b[2*4 + i], b[3*4 + i]]
        elif direction == 1: line = [b[i*4 + 3], b[i*4 + 2], b[i*4 + 1], b[i*4 + 0]]
        elif direction == 2: line = [b[3*4 + i], b[2*4 + i], b[1*4 + i], b[0*4 + i]]
        else:                line = [b[i*4 + 0], b[i*4 + 1], b[i*4 + 2], b[i*4 + 3]]

        slid, s = slide_line(line)
        total_score += s

        for j in range(4):
            if direction == 0:   r, c = j, i
            elif direction == 1: r, c = i, 3 - j
            elif direction == 2: r, c = 3 - j, i
            else:                r, c = i, j

            idx = r * 4 + c
            if new_b[idx] != slid[j]:
                moved = True
            new_b[idx] = slid[j]

    return new_b, total_score, moved

def spawn_tile(b):
    empty = [i for i in range(16) if b[i] == 0]
    if not empty:
        return b
    idx = random.choice(empty)
    b[idx] = 1 if random.random() < 0.9 else 2
    return b

def create_board():
    b = [0] * 16
    spawn_tile(b)
    spawn_tile(b)
    return b

## n-tuple pattern definitions and d4 dihedral symmetries

In [ ]:
BASE_TUPLES = [
    [0, 1, 2, 3, 4], # top row + corner support tile
    [0, 1, 4, 5, 2], # corner 2x2 + right extension
    [0, 1, 4, 5, 8], # corner 2x2 + down extension
    [0, 1, 2, 4, 8]  # corner L-corridor
]

SYMMETRIES = [
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    [3, 7, 11, 15, 2, 6, 10, 14, 1, 5, 9, 13, 0, 4, 8, 12],
    [15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
    [12, 8, 4, 0, 13, 9, 5, 1, 14, 10, 6, 2, 15, 11, 7, 3],
    [3, 2, 1, 0, 7, 6, 5, 4, 11, 10, 9, 8, 15, 14, 13, 12],
    [12, 13, 14, 15, 8, 9, 10, 11, 4, 5, 6, 7, 0, 1, 2, 3],
    [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15],
    [15, 11, 7, 3, 14, 10, 6, 2, 13, 9, 5, 1, 12, 8, 4, 0]
]

EXPANDED_TUPLES = []
for b in range(4):
    for s in range(8):
        sym = SYMMETRIES[s]
        base = BASE_TUPLES[b]
        EXPANDED_TUPLES.append((
            b,
            sym[base[0]],
            sym[base[1]],
            sym[base[2]],
            sym[base[3]],
            sym[base[4]]
        ))

NUM_BASE_TUPLES = 4
ENTRIES_PER_TUPLE = 1048576 # 16^5
TOTAL_WEIGHTS = NUM_BASE_TUPLES * ENTRIES_PER_TUPLE

print(f'expanded symmetric tuples: {len(EXPANDED_TUPLES)}')
print(f'total weights: {TOTAL_WEIGHTS:,} float32 values ({TOTAL_WEIGHTS * 4 / (1024 * 1024):.2f} mb)')

weights = np.zeros((NUM_BASE_TUPLES, ENTRIES_PER_TUPLE), dtype=np.float32)

## afterstate evaluation and exact normalized td updates

In [ ]:
def evaluate_afterstate(b, w=weights):
    val = 0.0
    for b_id, c0, c1, c2, c3, c4 in EXPANDED_TUPLES:
        idx = (b[c0] << 16) | (b[c1] << 12) | (b[c2] << 8) | (b[c3] << 4) | b[c4]
        val += w[b_id, idx]
    return val

def update_afterstate_td(afterstate, delta, alpha, w=weights):
    counts = {}
    for b_id, c0, c1, c2, c3, c4 in EXPANDED_TUPLES:
        idx = (afterstate[c0] << 16) | (afterstate[c1] << 12) | (afterstate[c2] << 8) | (afterstate[c3] << 4) | afterstate[c4]
        pair = (b_id, idx)
        counts[pair] = counts.get(pair, 0) + 1
    
    norm = sum(c * c for c in counts.values())
    step = (alpha * delta) / norm
    
    for (b_id, idx), c in counts.items():
        w[b_id, idx] += c * step

## temporal difference reinforcement learning training loop (100,000 games)

In [ ]:
TOTAL_GAMES = 100000
REPORT_INTERVAL = 2500

ALPHA_START = 0.05
ALPHA_END = 0.002

scores_history = []
max_tiles_history = []
recent_scores = []
recent_tiles = []

t_start = time.time()
print(f'starting td self-play training across {TOTAL_GAMES:,} games...')

for g in range(1, TOTAL_GAMES + 1):
    alpha = ALPHA_END + (ALPHA_START - ALPHA_END) * (1.0 - g / TOTAL_GAMES)
    b = create_board()
    game_score = 0
    prev_afterstate = None

    while True:
        best_val = -1e9
        best_after = None
        best_score = 0
        best_dir = -1

        for d in range(4):
            after, s, moved = move_board(b, d)
            if not moved:
                continue
            v = s + evaluate_afterstate(after)
            if v > best_val:
                best_val = v
                best_after = after
                best_score = s
                best_dir = d

        if best_dir == -1:
            if prev_afterstate is not None:
                delta = 0.0 - evaluate_afterstate(prev_afterstate)
                update_afterstate_td(prev_afterstate, delta, alpha)
            break

        if prev_afterstate is not None:
            target = best_score + evaluate_afterstate(best_after)
            delta = target - evaluate_afterstate(prev_afterstate)
            update_afterstate_td(prev_afterstate, delta, alpha)

        game_score += best_score
        prev_afterstate = best_after
        b = list(best_after)
        spawn_tile(b)

    scores_history.append(game_score)
    max_t = 1 << max(b)
    max_tiles_history.append(max_t)
    recent_scores.append(game_score)
    recent_tiles.append(max_t)

    if g % REPORT_INTERVAL == 0:
        elapsed = time.time() - t_start
        rate = g / elapsed
        eta_sec = (TOTAL_GAMES - g) / rate if rate > 0 else 0
        avg_s = np.mean(recent_scores[-REPORT_INTERVAL:])
        max_s = np.max(recent_scores[-REPORT_INTERVAL:])
        r2048 = np.mean([t >= 2048 for t in recent_tiles[-REPORT_INTERVAL:]]) * 100
        r4096 = np.mean([t >= 4096 for t in recent_tiles[-REPORT_INTERVAL:]]) * 100
        r8192 = np.mean([t >= 8192 for t in recent_tiles[-REPORT_INTERVAL:]]) * 100

        print(f'games: {g:6d}/{TOTAL_GAMES} | avg score: {avg_s:6.0f} | max: {max_s:6.0f} | '
              f'2048: {r2048:4.1f}% | 4096: {r4096:4.1f}% | 8192: {r8192:4.1f}% | '
              f'rate: {rate:4.1f} g/s | elapsed: {elapsed/60:4.1f}m | eta: {eta_sec/60:4.1f}m')

t_total = time.time() - t_start
print(f'\ntraining complete in {t_total:.1f}s ({t_total/60:.1f} minutes).')

## evaluation and developer benchmark challenge verification (1,000 games)

In [ ]:
EVAL_GAMES = 1000
print(f'evaluating trained n-tuple weights on {EVAL_GAMES} clean test games...')

eval_scores = []
eval_tiles = []

for _ in range(EVAL_GAMES):
    b = create_board()
    game_score = 0
    while True:
        best_val = -1e9
        best_after = None
        best_score = 0
        best_dir = -1

        for d in range(4):
            after, s, moved = move_board(b, d)
            if not moved:
                continue
            v = s + evaluate_afterstate(after)
            if v > best_val:
                best_val = v
                best_after = after
                best_score = s
                best_dir = d

        if best_dir == -1:
            break
        game_score += best_score
        b = list(best_after)
        spawn_tile(b)

    eval_scores.append(game_score)
    eval_tiles.append(1 << max(b))

tile_distribution = {}
for t in sorted(set(eval_tiles)):
    count = eval_tiles.count(t)
    tile_distribution[int(t)] = f'{count} ({count / EVAL_GAMES * 100:.1f}%)'

mean_score = float(np.mean(eval_scores))
median_score = float(np.median(eval_scores))
max_score = int(np.max(eval_scores))

print('\nevaluation results:')
print(f'  mean score:   {mean_score:,.0f}')
print(f'  median score: {median_score:,.0f}')
print(f'  max score:    {max_score:,}')
print(f'  developer benchmark: 144,868 points (verified in v3 challenge)')
print(f'  highest tile achieved: {max(eval_tiles):,}')
print(f'  tile distribution: {tile_distribution}')

## binary export and packaging

In [ ]:
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. export binary weights table (tuples_v3.bin)
binary_filename = 'tuples_v3.bin'
binary_path = os.path.join(OUTPUT_DIR, binary_filename)

weights_flat = weights.astype(np.float32)
weights_flat.tofile(binary_path)

file_size = os.path.getsize(binary_path)
print(f'exported {binary_filename} ({file_size:,} bytes, {file_size / (1024*1024):.2f} mb)')
assert file_size == TOTAL_WEIGHTS * 4, f'unexpected file size: {file_size}'

# 2. export training stats json
stats = {
    'total_games_trained': TOTAL_GAMES,
    'eval_games': EVAL_GAMES,
    'mean_score': mean_score,
    'median_score': median_score,
    'max_score': max_score,
    'tile_distribution': tile_distribution,
    'export_file': binary_filename,
    'num_weights': TOTAL_WEIGHTS,
    'file_size_bytes': file_size
}

stats_path = os.path.join(OUTPUT_DIR, 'training_stats.json')
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2)
print('saved training_stats.json')

# 3. create zip archive for kaggle download
zip_name = '02_model_training_output'
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(f'{zip_name}.zip')
print(f'\ndone. download {zip_name}.zip ({zip_size / (1024*1024):.2f} mb)')